In [ ]:
using MPSDynamics, ChaosTools, ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi
ω_b_0 = 7.0515*2*pi

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.155
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.15
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi *1.5


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.3135412
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.124276*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
t_max = 30/κ_b
ϕ_a_0, q_a_0=0.0, 0.0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)

system_prepared = prepare(system)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)
u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
u_0_new = vcat(u_0,u_dot_0)

drive = DriveTwoModes(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)
p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

problem = CoupledODEs(equation_1ODE!, u_0_new, p)
t_w = 60/κ_b
lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)

# Check chaos for Mean-field

In [ ]:
t_max = 30/κ_b
ϕ_a_0, q_a_0=0.0, 0.0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)

system_prepared = prepare(system)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)
u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
u_0_new = vcat(u_0,u_dot_0);

In [ ]:
φ_a_list = range(start = 0.01, stop = 0.25, step=0.0025)
φ_b_list = range(start = 0.01, stop = 0.5, step=0.005)

In [ ]:
Lyap_coef_chaos = Array{Float64}(undef, length(φ_b_list), length(φ_a_list))
g_2_matrix = Array{Float64}(undef, length(φ_b_list), length(φ_a_list))
t_w = 60/κ_b
for i in ProgressBar(1:length(φ_a_list))
    φ_a = φ_a_list[i]
    Z_a = 2*ϕ_quantum^2*φ_a^2
    C_a_inv = Z_a*ω_a
    E_Ca = C_a_inv/2
    
    for j in 1:length(φ_b_list)
        φ_b = φ_b_list[j]
        g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b
        if g2/2/pi*1e3 < 50 && g2/2/pi*1e3 > 20
            Z_b = 2*ϕ_quantum^2*φ_b^2
            C_b_inv = Z_b*ω_b_0
            E_Cb = C_b_inv/2
            system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
            system_prepared = prepare(system)
            g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
            ϵ_d=10.124276*g2
            drive = DriveTwoModes(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
            drive_prepared = prepare(drive)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            problem = CoupledODEs(equation_1ODE!, u_0_new, p)

            Lyap_coef_chaos[j,i] = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
        else
            Lyap_coef_chaos[j,i] = NaN
        end
        g_2_matrix[j,i] = g2
    end
end

In [ ]:
φ_a_lis = 0.01:0.01:maximum(φ_a_list)

heatmap(φ_a_list.^2,φ_b_list,Lyap_coef_chaos,cmap = :bluesreds)
#for φ_b in φ_b_list
#    plot!(φ_a_lis, (minimum(φ_a_list)./φ_a_lis).^2 .*φ_b)
#end
plot!(xlabel=L"(\phi^a_{ZPF})^2", ylabel=L"\phi^b_{ZPF}",
    size=(size_px_car[1],size_px_car[2]),
    colorbar_title="\nLyapunov exp.", title = "Mean field "*L"κ_b t= "*"$(round(κ_b*t_max))"*", "*L"κ_b t_W="*"$(round(κ_b*t_w))",
    clims=(minimum(filter(!isnan,Lyap_coef_chaos)),-minimum(filter(!isnan,Lyap_coef_chaos))),
    right_margin=10mm)
#savefig("../../figures/Two-photon/heatmap_lyapunov_coef.pdf")

In [ ]:
(besselj(1,ϵ_0)*E_J*0.18^2*0.16)/2/pi*1e3

In [ ]:

heatmap(φ_a_list.^2,φ_b_list,g_2_matrix/2/pi*1e3)
plot!(xlabel=L"(\phi^a_{ZPF})^2", ylabel=L"\phi^b_{ZPF}",
    size=(size_px_car[1],size_px_car[2]),
    colorbar_title="\n"*L"g_2/2\pi"*" (MHz)", title = "Mean field "*L"κ_b t= "*"$(round(κ_b*t_max))"*", "*L"κ_b t_W="*"$(round(κ_b*t_w))",
    clims=(minimum(g_2_matrix/2/pi*1e3),maximum(g_2_matrix/2/pi*1e3)),
    right_margin=10mm)
#savefig("../../figures/Two-photon/heatmap_lyapunov_coef.pdf")

In [ ]:
Lyap_coef_chaos_sign_1 = map(x -> x>0 ? 1 : x, Lyap_coef_chaos)
Lyap_coef_chaos_sign = map(x -> x<0 ? -1 : x, Lyap_coef_chaos_sign_1);

In [ ]:
φ_a_lis = 0.01:0.01:maximum(φ_a_list)

heatmap(φ_a_list.^2,φ_b_list,Lyap_coef_chaos_sign, cmap=:bluesreds)
#for φ_b in φ_b_list
#    plot!(φ_a_lis, (minimum(φ_a_list)./φ_a_lis).^2 .*φ_b)
#end
plot!(xlabel=L"(\phi^a_{ZPF})^2", ylabel=L"\phi^b_{ZPF}",
    size=(size_px_car[1],size_px_car[2]),
    colorbar_title="\nLyapunov exp.", title = "Mean field "*L"κ_b t= "*"$(round(κ_b*t_max))"*", "*L"κ_b t_W="*"$(round(κ_b*t_w))",
    clims=(minimum(Lyap_coef_chaos_sign),maximum(Lyap_coef_chaos_sign)),
    right_margin=10mm)
scatter!((0.033,0.17), ms=1)
#savefig("../../figures/Two-photon/heatmap_lyapunov_coef.pdf")

In [ ]:
sqrt(0.033)